# NB3 — FastAPI `/search` Endpoint + Latency Benchmark

**Stack:** FastAPI + Starlette TestClient + Searcher từ `app/search.py`.
Maps to slide §7 (Production Patterns) + deliverable bullets 1, 4.

> Mục tiêu: bọc `Searcher` thành REST API, đo P50/P95/P99 latency, đảm bảo
> P99 < 50 ms cho hybrid mode (rubric threshold).

In [1]:
import _setup  # noqa: F401
import statistics
import time
import json
from pathlib import Path

## 1. Khởi động API server (in-process TestClient)

Dùng Starlette TestClient để test FastAPI app in-process. Searcher được build
1 lần rồi tái dùng toàn bộ benchmark — cùng behavior như production.

In [2]:
ROOT = Path(_setup.__file__).resolve().parent.parent
import sys
sys.path.insert(0, str(ROOT))

from starlette.testclient import TestClient
from app.main import app

print("Building Searcher (embeds 1000 docs, may take ~5 min on CPU)...")
t_start = time.perf_counter()
# TestClient's context manager triggers lifespan startup (Searcher.from_corpus)
client = TestClient(app)
client.__enter__()
build_time = time.perf_counter() - t_start
print(f"Searcher ready in {build_time:.1f}s")

r = client.get("/healthz")
print("healthz:", r.json())

/tmp/ipykernel_180620/2395657552.py:5: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient


/home/nguyen-xuan-quan/vinai/labs/Track2-Day19-2A202601976-NguyenXuanQuan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building Searcher (embeds 1000 docs, may take ~5 min on CPU)...


Searcher ready in 200.0s
healthz: {'ready': True, 'n_docs': 1000}


## 2. Single query — kiểm tra response shape

In [3]:
r = client.get("/search", params={"q": "cloud computing tự động mở rộng", "mode": "hybrid"})
r.raise_for_status()
body = r.json()
print(f"latency_ms: {body['latency_ms']:.1f}")
print(f"top-3 hits:")
for h in body["hits"][:3]:
    print(f"  {h['doc_id']:>14}  score={h['score']:.4f}  {h['title']}")

latency_ms: 34.7
top-3 hits:
       cloud_016  score=0.0325  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_072  score=0.0323  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_053  score=0.0315  Điện toán đám mây: tự động mở rộng theo lưu lượng


## 3. Latency benchmark (100 queries × 3 modes)

50 golden queries × 2 reps = 100 calls/mode. Latency từ `body["latency_ms"]`
(server-side measurement, không tính TestClient overhead).

In [4]:
DATA = ROOT / "data"
golden = [json.loads(l) for l in (DATA / "golden_set.jsonl").open(encoding="utf-8")]

# Warmup: 10 queries per mode per README troubleshooting tip
# "Bình thường ở cold start. Chạy 10 query warmup trước rồi đo lại."
print("Running warmup (10 queries each mode)...")
for mode in ("keyword", "semantic", "hybrid"):
    for q in golden[:10]:
        client.get("/search", params={"q": q["query"], "mode": mode})
print("Warmup done.")


def percentile(values: list[float], p: float) -> float:
    n = len(values)
    if n == 0:
        return 0.0
    return sorted(values)[min(int(n * p), n - 1)]


def benchmark_mode(mode: str, reps: int = 2) -> dict[str, float]:
    server_latencies: list[float] = []
    wall_latencies: list[float] = []
    for _ in range(reps):
        for q in golden:
            t0 = time.perf_counter()
            r = client.get("/search", params={"q": q["query"], "mode": mode})
            wall_latencies.append((time.perf_counter() - t0) * 1000)
            server_latencies.append(r.json()["latency_ms"])
    return {
        "p50_server": percentile(server_latencies, 0.50),
        "p95_server": percentile(server_latencies, 0.95),
        "p99_server": percentile(server_latencies, 0.99),
        "p99_wall":   percentile(wall_latencies, 0.99),
    }


print(f"  {'mode':10}  {'P50':>7}  {'P95':>7}  {'P99':>7}  {'P99(wall)':>9}")
results = {}
for mode in ("keyword", "semantic", "hybrid"):
    res = benchmark_mode(mode)
    results[mode] = res
    print(f"  {mode:10}  {res['p50_server']:>5.1f}ms  {res['p95_server']:>5.1f}ms  "
          f"{res['p99_server']:>5.1f}ms  {res['p99_wall']:>7.1f}ms")

Running warmup (10 queries each mode)...


Warmup done.
  mode            P50      P95      P99  P99(wall)
  keyword       0.9ms    1.3ms    2.1ms      3.2ms


  semantic      1.0ms   43.3ms   50.4ms     51.5ms


  hybrid        2.5ms    3.1ms    3.4ms      4.4ms


## 4. Rubric assertion — hybrid P99 server-side < 50ms

In [5]:
hybrid_p99 = results["hybrid"]["p99_server"]
print(f"Hybrid P99 server-side: {hybrid_p99:.1f}ms")
if hybrid_p99 < 50:
    print(f"PASS — hybrid P99 < 50ms ({hybrid_p99:.1f}ms)")
else:
    print(f"WARN — hybrid P99 >= 50ms ({hybrid_p99:.1f}ms)")
    print("  Possible causes: cold cache, fastembed model not warm yet")

Hybrid P99 server-side: 3.4ms
PASS — hybrid P99 < 50ms (3.4ms)


## 5. Cleanup

In [6]:
client.__exit__(None, None, None)
print("API TestClient closed")

API TestClient closed


## Deliverable evidence

1. Output cell 2: healthz ready + single hybrid query response with `top-3 hits`.
2. Output cell 3: latency table P50/P95/P99 for keyword/semantic/hybrid.
3. Output cell 4: hybrid P99 < 50ms PASS.

---

## Vibe-coding callout

**Delegate freely:** FastAPI scaffolding, Pydantic response model, lifespan
handler. AI generates this perfectly from spec.

**Think hard yourself:** what to measure — server-side vs wall-clock, P99 vs P50.
These are judgement decisions; don't ask AI to pick the metric.